# Pump prediction research

Run `poetry install`, then `poetry run jupyter lab` from the repository root. Restart the kernel and run all cells in order. The seven models use the saved Optuna studies; this notebook refits them without starting a new search. Setup and retuning instructions are in [reproduction guide](../docs/reproduction.md#training-and-evaluation).

Events are split chronologically into **60% training / 10% tuning and early stopping / 20% comparison validation / 10% test**. Whole cross-sections and simultaneous events stay together. Validation Top@k%AUC selects the model before test evaluation. These periods were inspected in earlier research, so results are retrospective.

All plots use Matplotlib with a fixed paper style. The displayed images are exported unchanged to `paper/src/images/`; rebuild the LaTeX paper to include them. Tables, complete bootstrap draws/comparisons and training logs go to `analysis_outputs/split_60_10_20_10_v1/deterministic_selection/`. Saved cell outputs are omitted to keep this notebook small.

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from functools import partial

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from pumps_and_dumps.core.paths import SQLITE_URL, get_root_dir
from pumps_and_dumps.backtest import evaluation
from pumps_and_dumps.backtest.datasets.sample import DatasetType

get_ipython().run_line_magic("run", str(get_root_dir() / "notebooks/analysis_helpers.ipynb"))

NOTEBOOK_DIR = get_root_dir() / "notebooks"
OUTPUT_DIR = NOTEBOOK_DIR / "analysis_outputs" / "split_60_10_20_10_v1" / "deterministic_selection"
IMAGES = OUTPUT_DIR / "images"
PAPER_IMAGES = get_root_dir() / "paper" / "src" / "images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STUDY_STORAGE = SQLITE_URL

plt.style.use("default")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10, "savefig.facecolor": "white"})
show_paper_figure = partial(show_figure, image_dir=IMAGES, paper_dir=PAPER_IMAGES)

## Data and features

Regressors end strictly before the decision time, 15 minutes before the announcement. The split table records the actual event counts and UTC dates.

In [ ]:
from pumps_and_dumps.backtest.datasets.build_dataset import create_dataset

df = create_dataset()
df["is_pumped"] = df["currency_pair"] == df["pumped_currency_pair"]

In [ ]:
from pumps_and_dumps.backtest.datasets.splitting import split_model_selection_datasets

pd.DataFrame([
    {"Partition": split.name, "Events": frame.pump_hash.nunique(),
     "First event (UTC)": frame.pump_time.min(), "Last event (UTC)": frame.pump_time.max()}
    for split, frame in split_model_selection_datasets(df).items()
]).set_index("Partition")

In [ ]:
show_paper_figure(plot_cross_section_sizes(df), "cs_size.jpeg")

In [ ]:
from pumps_and_dumps.backtest.pipelines.base_pipeline import add_col_pump_id, cross_section_standardisation
from pumps_and_dumps.core.feature_type import FeatureType
from pumps_and_dumps.core.time_utils import NamedTimeDelta

df, feature_summary = describe_features(df)
feature_summary.to_csv(OUTPUT_DIR / "feature_summary.csv")
df_scaled = cross_section_standardisation(add_col_pump_id(df.copy()))
show_paper_figure(
    plot_feature_distributions(
        df_raw=df, df_scaled=df_scaled,
        feature_types=[FeatureType.ASSET_RETURN, FeatureType.ASSET_RETURN_ZSCORE, FeatureType.QUOTE_ABS_ZSCORE],
        offset=NamedTimeDelta.ONE_DAY, n_pumps=5, random_state=42,
    ),
    "cs_standardization_impact.png",
)

## Fit the seven candidates

Reuse the saved tuning parameters and fixed model seeds. Verbose fit output is saved in `training.log`.

In [ ]:
from pumps_and_dumps.backtest.pipelines.logistic_regression.pipeline import LogisticRegressionPipeline
from pumps_and_dumps.backtest.pipelines.random_forest.pipeline import RandomForestPipeline
from pumps_and_dumps.backtest.pipelines.catboost_classifier.pipeline import CatboostClassifierPipeline
from pumps_and_dumps.backtest.pipelines.catboost_smote.pipeline import CatboostClassifierSMOTEPipeline
from pumps_and_dumps.backtest.pipelines.catboost_ranker.pipeline import CatboostRankerPipeline
from pumps_and_dumps.backtest.pipelines.catboost_topk_auc.pipeline import CatboostClassifierTOPKAUCPipeline
from pumps_and_dumps.backtest.pipelines.catboost_top5.pipeline import CatboostTop5Pipeline

candidates = [
    (LogisticRegressionPipeline, "Logistic regression"),
    (RandomForestPipeline, "Random forest"),
    (CatboostClassifierPipeline, "CatBoost classifier"),
    (CatboostClassifierSMOTEPipeline, "CatBoost classifier + SMOTE"),
    (CatboostRankerPipeline, "CatBoost ranker"),
    (CatboostClassifierTOPKAUCPipeline, "CatBoost classifier + Top@k%AUC early stopping"),
    (CatboostTop5Pipeline, "CatBoost + Top-5"),
]

with (OUTPUT_DIR / "log").open("w") as log, redirect_stdout(log), redirect_stderr(log):
    experiments = [
        train_experiment(pipeline(study_storage=STUDY_STORAGE), name, tuned=True)
        for pipeline, name in candidates
    ]

experiment_cb_topkpauc = experiments[5]
print(f"Fitted {len(experiments)} candidates; details in {OUTPUT_DIR / 'log'}")

In [ ]:
show_paper_figure(plot_feature_importances(experiment_cb_topkpauc), "feature_importances.png")

## Validation and model selection

Top@k is the share of pumps whose manipulated asset ranks among the top k; Top@k% uses `ceil(k% × candidate count)`. Top@k%AUC integrates the 0–20% hit curve on a 0.5-percentage-point grid and normalizes its area.

Use **10,000 paired whole-pump bootstrap draws, seed 42**, with predictions fixed and the same sampled events for every model. Intervals are 95% pointwise percentiles. Two-sided centered-null bootstrap tests use an add-one correction; Holm adjustment covers all 21 model pairs on validation. Significance comes from paired differences, not overlap of marginal intervals. Threshold intervals and full pairwise results are exported as CSVs.

In [ ]:
from pumps_and_dumps.backtest.robustness.validation_bootstrap import (
    evaluate_validation_bootstrap,
)

N_BOOTSTRAP = 10_000
BOOTSTRAP_ALPHA = 0.05
BOOTSTRAP_SEED = 42
BOOTSTRAP_OUTPUT = OUTPUT_DIR / "significance"
BOOTSTRAP_IMAGES = BOOTSTRAP_OUTPUT / "images"
BOOTSTRAP_IMAGES.mkdir(parents=True, exist_ok=True)

validation_bootstrap = evaluate_validation_bootstrap(
    experiments,
    n_bootstrap=N_BOOTSTRAP,
    alpha=BOOTSTRAP_ALPHA,
    random_state=BOOTSTRAP_SEED,
)
report_validation_bootstrap(
    validation_bootstrap,
    NOTEBOOK_DIR,
    n_bootstrap=N_BOOTSTRAP,
    output_dir=BOOTSTRAP_OUTPUT / "validation",
)

In [ ]:
show_figure(
    plot_model_bootstrap_comparison(validation_bootstrap.auc_summary, validation_bootstrap.pairwise),
    "validation_bootstrap_model_comparison.png",
    image_dir=BOOTSTRAP_IMAGES,
)

In [ ]:
selected_experiment, validation_auc_ranking = evaluation.select_best_experiment_on_validation(
    experiments, bootstrap_results=validation_bootstrap
)
print(f"Selected by validation Top@k%AUC: {selected_experiment.get_experiment_name()}")
validation_metrics = report_model_metrics(
    experiments, DatasetType.VALIDATION, output_dir=OUTPUT_DIR, paper_dir=PAPER_IMAGES
)

## Test evaluation of frozen models

Keep the validation-selected model fixed. The same bootstrap resamples only test pumps, with replacement; repeated events contribute their complete cross-sections again. Score each fitted model once for the bootstrap and reuse predictions across draws.

Report Top@k%AUC, precision, recall and F1. At Top@1, with one true asset per event, the three classification metrics coincide. Holm correction covers **all 84 model-pair × metric hypotheses together**, including those coincident metrics. The comparison plot uses these original adjusted decisions; full metric distributions, predictions, draw IDs, intervals and all comparisons are saved as CSVs. Only rejected hypotheses are displayed below to avoid repeating 84 rows.

Inference conditions on fitted models and assumes independent, exchangeable pump events; it excludes retraining uncertainty. An inconclusive comparison does not establish equivalence or a unique winner. Test outcomes do not change the selected model or portfolio rule.

In [ ]:
test_metrics = report_model_metrics(
    experiments, DatasetType.TEST, output_dir=OUTPUT_DIR, paper_dir=PAPER_IMAGES
)

In [ ]:
from pumps_and_dumps.backtest.robustness.metric_bootstrap import evaluate_test_metric_bootstrap

N_TEST_BOOTSTRAP = 10_000
TEST_BOOTSTRAP_ALPHA = 0.05
TEST_BOOTSTRAP_SEED = 42
CLASSIFICATION_TOP_K = 1
TEST_BOOTSTRAP_OUTPUT = OUTPUT_DIR / "test_pump_bootstrap"

test_bootstrap = evaluate_test_metric_bootstrap(
    experiments,
    n_bootstrap=N_TEST_BOOTSTRAP,
    alpha=TEST_BOOTSTRAP_ALPHA,
    random_state=TEST_BOOTSTRAP_SEED,
    classification_top_k=CLASSIFICATION_TOP_K,
)
test_bootstrap_files = export_metric_bootstrap(test_bootstrap, TEST_BOOTSTRAP_OUTPUT)
print(
    f"{N_TEST_BOOTSTRAP:,} paired resamples × "
    f"{test_bootstrap.sampled_events.shape[1]} test pumps per sample × "
    f"{len(experiments)} fixed models."
)
print(f"Draw IDs, predictions, metric distributions and comparisons: {TEST_BOOTSTRAP_OUTPUT}")
confidence_level = 1 - TEST_BOOTSTRAP_ALPHA
display(
    test_bootstrap.metric_summary.assign(
        metric=lambda table: table["metric"].map(METRIC_LABELS)
    ).set_index(["metric", "model"])[
        ["point_estimate", "ci_lower", "ci_upper"]
    ].style.format(precision=4).set_caption(
        f"Original-test scores and {confidence_level:.0%} pointwise bootstrap intervals"
    )
)

In [ ]:
# Preserve the Holm correction over all four metrics; do not retest this subset.
test_auc_summary = test_bootstrap.metric_summary.query("metric == 'topk_percent_auc'").set_index("model")
test_auc_pairwise = test_bootstrap.pairwise.query("metric == 'topk_percent_auc'")
test_bootstrap_comparison = plot_model_bootstrap_comparison(test_auc_summary, test_auc_pairwise)
show_figure(
    test_bootstrap_comparison, "test_bootstrap_model_comparison.png",
    image_dir=TEST_BOOTSTRAP_OUTPUT, paper_dir=PAPER_IMAGES,
)

In [ ]:
print(f"Holm correction uses all {len(test_bootstrap.pairwise)} model-pair × metric tests together.")
print(f"Model frozen on comparison validation: {selected_experiment.get_experiment_name()}")
significant_test_pairs = test_bootstrap.pairwise.loc[test_bootstrap.pairwise["reject_h0"], [
    "metric", "model_a", "model_b", "observed_diff", "p_value_holm", "better_model",
]]
if significant_test_pairs.empty:
    print("No pairwise differences reject at the adjusted 5% level.")
else:
    display(significant_test_pairs.style.format({"observed_diff": "{:+.4f}", "p_value_holm": "{:.4f}"}))

## Test portfolios and impact sensitivity

Use equal weights in the validation-selected model's top K assets; **K=5** is primary and other sizes are descriptive. Entry is the first trade in `[T−15min, T)`; non-pumped holdings exit at or after T and the manipulated asset at or after T+1min. Missing fills retain their cash allocation. Returns use fixed event notional, USDT conversion and 25bp round-trip costs. BTC buy-and-hold uses the same time interval. Impact sensitivity is reported separately.

In [ ]:
# Fix these settings before inspecting test portfolio outcomes.
PORTFOLIO_SIZE = 5
PORTFOLIO_SIZES = [1, 2, 5, 10, 20, 30]
df_with_btc, comparison_stats = portfolio_benchmark(
    selected_experiment,
    portfolio_sizes=PORTFOLIO_SIZES,
)
comparison_stats

In [ ]:
df_with_btc.to_csv(OUTPUT_DIR / "tables" / "portfolio_returns.csv")
comparison_stats.to_csv(OUTPUT_DIR / "tables" / "portfolio_statistics.csv")
show_paper_figure(
    plot_equity_curves(
        df_with_btc, title="Test cumulative return by K vs time-aligned BTCUSDT buy-and-hold",
        baseline_cols=["BTCUSDT buy-and-hold"],
    ),
    "portfolios_vs_btc_test.png",
)

In [ ]:
from datetime import timedelta
from pumps_and_dumps.backtest.portfolio.topk import evaluate_topk_pnl_for_quantities

impact_quantities_usdt = [100, 200, 500, 1000, 2000, 5000, 10000]
dataset_test = selected_experiment.get_sample().get_dataset(DatasetType.TEST)
df_pnl_by_quantity = evaluate_topk_pnl_for_quantities(
    model=selected_experiment.get_model(),
    dataset=dataset_test,
    portfolio_size=PORTFOLIO_SIZE,
    quantities_usdt=impact_quantities_usdt,
    buy_before=timedelta(minutes=15),
    sell_after=timedelta(minutes=1),
    impact_lookback_days=14,
)
df_pnl_by_quantity[["quantity_usdt", "cumulative_roe"]]

In [ ]:
df_pnl_by_quantity.to_csv(OUTPUT_DIR / "tables" / "impact_sensitivity.csv", index=False)
show_paper_figure(plot_pnl_sensitivity(df_pnl_by_quantity), "pnl_vs_quantity_with_impact.png")

The following best-fitting entry and exit examples illustrate the impact model; they are descriptive selections, not model-selection evidence. Each fit uses observations strictly before the transaction it prices.

In [ ]:

for finder, filename in (
    (find_best_impact_example, "impact_regression_example.png"),
    (find_best_exit_impact_example, "exit_impact_regression_example.png"),
):
    show_paper_figure(show_impact_example(finder(dataset_test)), filename)

## Robustness

Refit the Top@k%AUC early-stopping pipeline on 25 seeded 70% subsets of TRAIN events and evaluate on fixed TEST events. This sensitivity analysis does not reselect the primary model. Feature-importance quantiles use the same runs; verbose output is saved in `robustness.log`.

In [ ]:
from pumps_and_dumps.backtest.robustness import subsets as robustness

ROBUSTNESS_OUTPUT = OUTPUT_DIR / "robustness"
with (OUTPUT_DIR / "robustness.log").open("w") as log, redirect_stdout(log), redirect_stderr(log):
    df_robustness = robustness.run_cross_section_subset_robustness(
        pipeline_factory=partial(
            CatboostClassifierTOPKAUCPipeline, study_storage=STUDY_STORAGE
        ),
        subset_fraction=0.7,
        n_runs=25,
        tuned=True,
        base_seed=42,
        topk_bins=[0.01, 0.02, 0.05, 0.1, 0.2],
        collect_feature_importances=True,
        output_path=ROBUSTNESS_OUTPUT / "catboost_topkauc_subset_runs.csv",
        feature_importance_output_path=ROBUSTNESS_OUTPUT
        / "catboost_topkauc_feature_importances.csv",
    )

In [ ]:
metric_cols = [
    "topk_percent@0.01",
    "topk_percent@0.02",
    "topk_percent@0.05",
    "topk_percent@0.1",
    "topk_percent@0.2",
]
robustness_summary = robustness.summarise_robustness_distribution(
    results=df_robustness,
    metric_cols=["topk_percent_auc", *metric_cols],
)
robustness_summary

In [ ]:
robustness_summary.to_csv(OUTPUT_DIR / "tables" / "robustness_summary.csv")
show_paper_figure(
    plot_robustness_distribution(df_robustness, metric_cols=metric_cols),
    "robustness_metrics_distribution.png",
)

In [ ]:
fi_summary = robustness.summarise_feature_importance_distribution(
    df_robustness.attrs["feature_importances"], quantiles=(0.1, 0.25, 0.5, 0.75, 0.9), top_n=15,
)
fi_summary.to_csv(OUTPUT_DIR / "tables" / "feature_importance_quantiles.csv")
fi_summary[["p10", "p25", "p50", "p75", "p90"]].style.format(precision=3)

Temporal sensitivity uses the prespecified 2022-06-01 cutoff and reports only subperiods passing the minimum-event guard (10 events).

In [ ]:
REGIME_DATE = "2022-06-01"
df_subperiod = robustness.evaluate_subperiod_metrics(
    model=selected_experiment.get_model(),
    dataset_df=dataset_test.all_data(),
    date_col="pump_time",
    split_date=REGIME_DATE,
    topk_bins=[0.01, 0.02, 0.05, 0.1, 0.2],
)
df_subperiod

## Reproducibility and leakage checks

Keep the feature files, split manifest, study database, seeds and software environment fixed to reproduce results. The paper's archived evidence is in `paper/reproducibility_20261016.zip`; complete rerun outputs remain outside this notebook.

Automated checks in `tests/leakage/` mutate future trades and labels, verify that train/tuning/test events remain separate, restrict imputation fallback to TRAIN, enforce the regressor allowlist and order, and keep portfolio choices independent of future returns. `tests/features/test_feature_writer_causality.py` also rebuilds full cross-sections after changing future data. These checks support the absence of the tested leakage paths; they are not proof against every possible bias. Regressor windows and universe membership end strictly before T−15min; post-announcement fields are labels/evaluation data only. See the paper for the full methods and limitations.

```bash
poetry run pytest tests/leakage tests/features/test_feature_writer_causality.py tests/backtest/research tests/backtest/robustness
```